# 1. Inspection

Taking a first look at the SMS spam dataset. Nothing gets changed here, I only want to know what is in the file.

In [1]:
import pandas as pd

In [2]:
# the file is not utf-8, so it has to be read as latin-1
df = pd.read_csv('data/spam.csv', encoding='latin-1')
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [3]:
print('Rows:', df.shape[0])
print('Columns:', df.shape[1])
print(df.columns.tolist())

Rows: 5572
Columns: 5
['v1', 'v2', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4']


In [4]:
df.dtypes

v1            str
v2            str
Unnamed: 2    str
Unnamed: 3    str
Unnamed: 4    str
dtype: object

In [5]:
df.isnull().sum()

v1               0
v2               0
Unnamed: 2    5522
Unnamed: 3    5560
Unnamed: 4    5566
dtype: int64

The names are `v1` and `v2`, and there are three `Unnamed` columns that are almost completely empty. Let me see what is in them.

In [6]:
extra_cols = ['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4']
extra_rows = df[df[extra_cols].notna().any(axis=1)]
print('Rows with something in the extra columns:', len(extra_rows))
extra_rows[extra_cols].head()

Rows with something in the extra columns: 50


,Unnamed: 2,Unnamed: 3,Unnamed: 4
95,PO Box 5249,"MK17 92H. 450Ppw 16""",NaN
281,the person is definitely special for u..... B...,why to miss them,"just Keep-in-touch\"" gdeve.."""
444,HOWU DOIN? FOUNDURSELF A JOBYET SAUSAGE?LOVE ...,NaN,NaN
671,"wanted to say hi. HI!!!\"" Stop? Send STOP to ...",NaN,NaN
710,"this wont even start........ Datz confidence..""",NaN,NaN


In [7]:
# one full example
row = df.loc[95]
print(row['v2'])
print(row['Unnamed: 2'], '|', row['Unnamed: 3'])

Your free ringtone is waiting to be collected. Simply text the password \MIX\" to 85069 to verify. Get Usher and Britney. FML
 PO Box 5249 |  MK17 92H. 450Ppw 16"


These are just pieces of long messages that got split at a comma or a quote when the file was made. They don't hold any new information.

In [8]:
print('Duplicate rows:', df.duplicated(subset=['v1', 'v2']).sum())

Duplicate rows: 403


In [9]:
# most repeated messages
df['v2'].value_counts().head(5)

v2
Sorry, I'll call later                                                                                                                                                                 30
I cant pick the phone right now. Pls send a message                                                                                                                                    12
Ok...                                                                                                                                                                                  10
Please call our customer service representative on FREEPHONE 0808 145 4742 between 9am-11pm as you have WON a guaranteed å£1000 cash or å£5000 prize!                                   4
Wen ur lovable bcums angry wid u, dnt take it seriously.. Coz being angry is d most childish n true way of showing deep affection, care n luv!.. kettoda manda... Have nice day da.     4
Name: count, dtype: int64

In [10]:
# does the same message ever have both labels?
labels_per_message = df.drop_duplicates(subset=['v1', 'v2']).groupby('v2')['v1'].nunique()
print('Messages with conflicting labels:', (labels_per_message > 1).sum())

Messages with conflicting labels: 0


In [11]:
print(df['v1'].unique())

counts = df['v1'].value_counts()
percent = (counts / len(df) * 100).round(1)
print(counts)
print(percent)

<ArrowStringArray>
['ham', 'spam']
Length: 2, dtype: str
v1
ham     4825
spam     747
Name: count, dtype: int64
v1
ham     86.6
spam    13.4
Name: count, dtype: float64


In [12]:
print('SPAM examples')
for msg in df[df['v1'] == 'spam']['v2'].sample(3, random_state=1):
    print('-', msg)

print()
print('HAM examples')
for msg in df[df['v1'] == 'ham']['v2'].sample(3, random_state=1):
    print('-', msg)

SPAM examples
- Marvel Mobile Play the official Ultimate Spider-man game (å£4.50) on ur mobile right now. Text SPIDER to 83338 for the game & we ll send u a FREE 8Ball wallpaper
- Thank you, winner notified by sms. Good Luck! No future marketing reply STOP to 84122 customer services 08450542832
- Free msg. Sorry, a service you ordered from 81303 could not be delivered as you do not have sufficient credit. Please top up to receive the service.

HAM examples
- Can you pls pls send me a mail on all you know about relatives coming to deliver here? All you know about costs, risks, benefits and anything else. Thanks.
- Yeah, probably but not sure. Ilol let u know, but personally I wuldnt bother, then again if ur goin to then I mite as well!!
- Were gonna go get some tacos


In [13]:
# strange characters
odd = df['v2'].str.contains(r'[^\x00-\x7f]')
print('Messages with non-ASCII characters:', odd.sum())
print(df[odd]['v2'].iloc[0])

Messages with non-ASCII characters: 481
FreeMsg Hey there darling it's been 3 week's now and no word back! I'd like some fun you up for it still? Tb ok! XxX std chgs to send, å£1.50 to rcv


## What I found

- 5,572 rows. The label is in `v1` and the message text is in `v2`. Neither has missing values.
- The three `Unnamed` columns are leftovers from split messages, so they can be dropped.
- 403 duplicate messages, and none of them have conflicting labels. They should be removed, otherwise the same message can end up in both the train and the test set.
- Labels are only `ham` and `spam`, but they are imbalanced (about 87% ham, 13% spam). I'll need a stratified split and should not rely on accuracy alone.
- Some messages have broken characters, for example `å£` instead of `£`.

## To do in preprocessing

- Keep only `v1` and `v2`, rename them to `label` and `message`
- Drop duplicates
- Convert the label to 0/1
- Clean the text (lowercase, remove strange characters and extra spaces)